# Customer Churn Early-Warning System

**Business question:** which customers are about to leave, how much revenue is at stake, and what should the retention team do about it?

Plan: diagnose where churn concentrates, predict it with XGBoost (recall-first), then translate scores into dollars of revenue at risk with save plays.

## 0. Get the data

IBM Telco Customer Churn, 7,043 customers, 21 columns. One command, no login needed.

In [ ]:
import urllib.request, os
os.makedirs("data", exist_ok=True)
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
path = "data/telco_churn.csv"
if not os.path.exists(path):
    urllib.request.urlretrieve(url, path)
    print("downloaded")
else:
    print("already there")

## 1. Load and inspect

Known quirks to watch for:
- `TotalCharges` has blank strings (customers with tenure 0)
- `customerID` is an identifier, never a feature
- `Churn` is Yes/No, the target

In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv("data/telco_churn.csv")
print(df.shape)
print(df.dtypes)
print("\nchurn rate:", (df["Churn"] == "Yes").mean().round(3))
print("\nblank TotalCharges:", (df["TotalCharges"] == " ").sum())

## 2. Clean

Convert `TotalCharges` to numeric (blanks become NaN, then 0 for tenure-0 customers), map `Churn` to 1/0. The reusable version lives in `src/clean.py`.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
df["Churn"] = (df["Churn"] == "Yes").astype(int)
df[["tenure", "MonthlyCharges", "TotalCharges", "Churn"]].describe().round(2)

## 3. EDA: where does churn concentrate?

Questions to answer with numbers:
1. Churn rate by contract type (month-to-month vs one/two year)?
2. Churn rate by internet service (fiber vs DSL vs none)?
3. Churn by tenure bands: is it front-loaded in the first months?
4. Do add-on services (tech support, online security) protect against churn?
5. Which single segment has the worst churn rate, and how big is it?

In [ ]:
for col in ["Contract", "InternetService", "PaymentMethod"]:
    print(df.groupby(col)["Churn"].agg(["mean", "count"]).round(3).sort_values("mean", ascending=False))
    print()
df["tenure_band"] = pd.cut(df["tenure"], [0, 6, 12, 24, 48, 73],
                             labels=["0-6", "7-12", "13-24", "25-48", "49-72"])
print(df.groupby("tenure_band", observed=True)["Churn"].agg(["mean", "count"]).round(3))

In [ ]:
# TODO: your turn. Answer questions 4 and 5, then plot the top 3 charts:
# churn by contract, churn by tenure band, churn by internet service.
# Each chart needs a one-line business read in markdown below it.

## 4. Feature engineering

One-hot encode categoricals, keep numerics as-is, stratified 80/20 split. `customerID` stays out of the model.

In [ ]:
from sklearn.model_selection import train_test_split
numeric = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
categorical = [c for c in df.columns if c not in numeric + ["Churn", "customerID", "tenure_band"]]
X = pd.get_dummies(df[numeric + categorical], columns=categorical, drop_first=True)
y = df["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, "churn rate train:", y_train.mean().round(3))

## 5. Model: XGBoost, recall-first

A missed churner costs the whole account; a false alarm costs a retention call. So we optimize for recall and judge with ROC-AUC, not accuracy. Baseline logistic regression first, then XGBoost with class weights.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report
from xgboost import XGBClassifier

bl = LogisticRegression(max_iter=1000, class_weight="balanced")
bl.fit(X_train, y_train)
print("baseline ROC-AUC:", roc_auc_score(y_test, bl.predict_proba(X_test)[:, 1]).round(3))

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
xgb = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                    subsample=0.8, colsample_bytree=0.8,
                    scale_pos_weight=neg / pos, random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train)
proba = xgb.predict_proba(X_test)[:, 1]
print("xgboost ROC-AUC:", roc_auc_score(y_test, proba).round(3))
print(classification_report(y_test, proba > 0.5))

## 6. Evaluate like a business: top-decile lift

If the retention team can only call 10 percent of customers, how much better is the model's top decile than picking at random?

In [ ]:
ranked = pd.DataFrame({"y": y_test.values, "p": proba}).sort_values("p", ascending=False)
n = len(ranked) // 10
lift = ranked["y"].iloc[:n].mean() / ranked["y"].mean()
print(f"top-decile churn rate: {ranked['y'].iloc[:n].mean():.1%}")
print(f"overall churn rate:    {ranked['y'].mean():.1%}")
print(f"lift: {lift:.2f}x")

# TODO: feature importance. Which 5 features drive the model? Do they match
# the EDA story (contract, tenure, fiber)? Note any surprises in markdown.

## 7. Revenue at risk

A probability is not a business metric. Multiply churn probability by monthly charges to rank accounts by dollars at stake.

In [ ]:
risk = pd.DataFrame({
    "churn_probability": proba,
    "monthly_charges": X_test.index.map(df["MonthlyCharges"]),
})
risk["revenue_at_risk"] = risk["churn_probability"] * risk["monthly_charges"]
print(f"total monthly revenue at risk (test set): ${risk['revenue_at_risk'].sum():,.0f}")
risk.sort_values("revenue_at_risk", ascending=False).head(10).round(2)

## 8. Save plays and next steps

TODO: write 3 save plays, one per top-risk segment from the EDA. Format: segment, trigger, offer, expected effect. Example: month-to-month fiber customers in months 0-6 get a contract-upgrade discount plus free tech support for 3 months.

Then copy your final numbers into the repo README's Problem/Approach/Impact sections. That README is what recruiters actually read.